# Colab setup: Google Drive workspace + data

Prepares a fresh Colab runtime only (no training). The same first cell is at the top of
`02_phase2_training.ipynb`, so you normally do not need this notebook: open the training notebook
and run it from the top. All steps live in `src/data/colab_setup.py`:

1. Install `ultralytics` / `albumentations` (pinned in `requirements.txt`).
2. Link `outputs/logs` and `outputs/runs` to `MyDrive/<DRIVE_DIR>/outputs/` (persist across sessions).
3. Copy `MyDrive/<DRIVE_DIR>/data.zip` to local disk and extract it into `data/`
   (once per runtime; local disk is much faster than Drive for many small files).
4. Restore the committed split lists (the archive's copies have CRLF line endings).
5. Point `path:` in every dataset YAML under `data/processed` to this machine.

Each Colab notebook gets its own runtime, so this runs again in every new session.

In [ ]:
# ---- Settings ----
DRIVE_DIR = "cse365_pothole"   # folder under MyDrive that contains data.zip
REPO_URL = "https://github.com/queso-gato1355/deep-learning-cse365.git"
BRANCH = "main"

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive

def sh(*cmd: str) -> None:
    """Run a command, streaming its output into the notebook; raise if it fails."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"`{' '.join(cmd)}` failed with exit code {p.returncode}; see the output above.")

drive.mount("/content/drive")
REPO = Path("/content/deep-learning-cse365")
if (REPO / ".git").exists():
    sh("git", "-C", str(REPO), "pull", "--ff-only")
else:
    sh("git", "clone", "-b", BRANCH, REPO_URL, str(REPO))
os.chdir(REPO)
sh(sys.executable, "-m", "src.data.colab_setup", "--drive-dir", DRIVE_DIR)

In [ ]:
!git log --oneline -3
!du -sh data/* 2>/dev/null
!df -h /content /content/drive | cat